# Wang 5-Stack CNN evaluation — 3 classes

Loads `best_model_wang_3class.keras` and the test set, generates predictions and computes the same metrics as `Evaluation_3class.ipynb` for direct comparison between the two models:

- **Global metrics**: accuracy, precision, recall and F1;
- **Classification report**: per-class precision, recall and F1;
- **3×3 confusion matrix**: absolute counts and row-normalized;
- **One-vs-rest ROC and Precision-Recall curves**: one curve per class;
- **Per-class accuracy** bar chart;
- **Per-audio analysis**: majority vote over the 11 segments of each audio.

Classes:
- `0`: light
- `1`: moderate
- `2`: intense

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    f1_score, accuracy_score, precision_score, recall_score)
from sklearn.preprocessing import label_binarize
from Multiclasse_data_pipeline import build_datasets, configure_gpu
from tensorflow.keras import backend as K

In [ ]:
MODEL_PATH  = "best_model_wang_3class.keras"
CSV_PATH    = "Split70-15-15_3class.csv"
BATCH_SIZE  = 64
CLASS_NAMES = ["light", "moderate", "intense"]
N_CLASSES   = 3


In [ ]:
def categorical_focal_loss(gamma=2.0, alpha=None):
    if alpha is not None:
        alpha_tensor = tf.constant(alpha, dtype=tf.float32)
    else:
        alpha_tensor = None

    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1.0 - K.epsilon())
        ce = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1.0 - p_t, gamma)
        focal_ce = focal_weight * ce
        if alpha_tensor is not None:
            focal_ce = focal_ce * alpha_tensor
        return tf.reduce_sum(focal_ce, axis=-1)

    return loss


In [ ]:
# Load model and datasets
configure_gpu()

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=categorical_focal_loss(gamma=2.0, alpha=[0.4, 0.4, 0.2]),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")],
)

print(f"Modelo carregado: {MODEL_PATH}")
print(f"Total de parâmetros: {model.count_params():,}")

_, _, test_ds = build_datasets(
    CSV_PATH, batch_size=BATCH_SIZE,
    label_mode="one_hot", mixup_train=False,
)


In [ ]:
# Test set metadata
full_df = pd.read_csv(CSV_PATH)
test_df = full_df[full_df["split"] == "test"].reset_index(drop=True)

print(f"Total de amostras no test (segmentos): {len(test_df)}")
print(f"\nDistribuição por classe (segmentos):")
print(test_df["class_name"].value_counts().to_string())
print(f"\nDistribuição por classe (áudios únicos):")
print(test_df.groupby("class_name")["audio_id"].nunique().to_string())


In [ ]:
# Test set predictions
print("Rodando inferência no test set...")

y_probs = model.predict(test_ds, verbose=1)
y_pred  = np.argmax(y_probs, axis=1)
y_true  = test_df["label"].values.astype(int)

assert len(y_pred) == len(y_true), (
    f"Tamanhos diferentes! predict={len(y_pred)}, csv={len(y_true)}. "
    "Verifique se o test_ds está em shuffle=False."
)

test_df["pred"] = y_pred
for i, name in enumerate(CLASS_NAMES):
    test_df[f"prob_{name}"] = y_probs[:, i]

print(f"\nDistribuição das predições:")
for i, name in enumerate(CLASS_NAMES):
    n = int((y_pred == i).sum())
    print(f"  - {name:>9}: {n}")


In [ ]:
# Global test metrics
test_accuracy = accuracy_score(y_true, y_pred)

test_prec_macro = precision_score(y_true, y_pred, average="macro",    zero_division=0)
test_rec_macro  = recall_score   (y_true, y_pred, average="macro",    zero_division=0)
test_f1_macro   = f1_score       (y_true, y_pred, average="macro",    zero_division=0)

test_prec_w = precision_score(y_true, y_pred, average="weighted", zero_division=0)
test_rec_w  = recall_score   (y_true, y_pred, average="weighted", zero_division=0)
test_f1_w   = f1_score       (y_true, y_pred, average="weighted", zero_division=0)

print("=" * 55)
print("MÉTRICAS FINAIS NO TEST SET — Wang 5-Stack CNN (por segmento)")
print("=" * 55)
print(f"Accuracy:                {test_accuracy:.4f}")
print()
print(f"Precision  (macro):      {test_prec_macro:.4f}")
print(f"Recall     (macro):      {test_rec_macro:.4f}")
print(f"F1-score   (macro):      {test_f1_macro:.4f}    <-- métrica principal")
print()
print(f"Precision  (weighted):   {test_prec_w:.4f}")
print(f"Recall     (weighted):   {test_rec_w:.4f}")
print(f"F1-score   (weighted):   {test_f1_w:.4f}")
print("=" * 55)

print("\nClassification report completo (precision/recall/F1 por classe):")
print(classification_report(y_true, y_pred,
                            target_names=CLASS_NAMES, digits=4))


In [ ]:
# 3×3 confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=True,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size": 14})
plt.xlabel("Predito", fontsize=12)
plt.ylabel("Real", fontsize=12)
plt.title("Matriz de Confusão (contagens) — Wang 5-Stack CNN — Test Set", fontsize=13)
plt.tight_layout()
plt.show()

print("Total por classe real:")
for i, name in enumerate(CLASS_NAMES):
    total   = int(cm[i].sum())
    correct = int(cm[i, i])
    print(f"  - {name:>9}: {correct}/{total} corretos")


In [ ]:
# Row-normalized 3×3 confusion matrix
cm_norm = confusion_matrix(y_true, y_pred, normalize="true")

plt.figure(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues", cbar=True,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size": 13})
plt.xlabel("Predito", fontsize=12)
plt.ylabel("Real", fontsize=12)
plt.title("Matriz de Confusão (normalizada) — Wang 5-Stack CNN — Test Set", fontsize=13)
plt.tight_layout()
plt.show()

print("Recall por classe (% de cada classe real corretamente classificada):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  - {name:>9}: {cm_norm[i, i]*100:.2f}%")


In [ ]:
# One-vs-rest ROC curves
y_true_onehot = label_binarize(y_true, classes=list(range(N_CLASSES)))

plt.figure(figsize=(8, 6))
colors = ["#55a868", "#c44e52", "#8172b2"]
for i, name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, color=colors[i],
             label=f"{name}  (AUC = {roc_auc:.4f})")

plt.plot([0, 1], [0, 1], "--", color="gray", label="Random (AUC = 0.5)")
plt.xlabel("False Positive Rate (1 - Especificidade)")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Curvas ROC — One-vs-Rest — Wang 5-Stack CNN — Test Set")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# One-vs-rest Precision-Recall curves
plt.figure(figsize=(8, 6))
colors = ["#55a868", "#c44e52", "#8172b2"]
for i, name in enumerate(CLASS_NAMES):
    prec, rec, _ = precision_recall_curve(y_true_onehot[:, i], y_probs[:, i])
    ap = average_precision_score(y_true_onehot[:, i], y_probs[:, i])
    plt.plot(rec, prec, linewidth=2, color=colors[i],
             label=f"{name}  (AP = {ap:.4f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curvas Precision-Recall — One-vs-Rest — Wang 5-Stack CNN — Test Set")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Per-class accuracy bar chart
per_class_acc = []
for i in range(N_CLASSES):
    mask = (y_true == i)
    acc  = (y_pred[mask] == y_true[mask]).mean() if mask.sum() > 0 else 0.0
    per_class_acc.append(acc * 100)

plt.figure(figsize=(8, 5))
colors = ["#55a868", "#c44e52", "#8172b2"]
bars = plt.bar(CLASS_NAMES, per_class_acc, color=colors)
plt.ylim([0, 105])
plt.ylabel("Acerto (%)")
plt.title("Acurácia por classe — Wang 5-Stack CNN — Test Set (segmentos)")
for bar, pct in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 1, f"{pct:.2f}%",
             ha="center", fontsize=11)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Per-audio analysis: majority vote of the 11 segments
audio_results = test_df.groupby("audio_id").agg(
    true_label    = ("label", "first"),
    pred_majority = ("pred",  lambda x: x.value_counts().idxmax()),
    class_name    = ("class_name", "first"),
    n_segments    = ("pred", "count"),
    agreement     = ("pred", lambda x: x.value_counts().iloc[0] / len(x)),
).reset_index()

print(f"Total de áudios no test: {len(audio_results)}")
print(f"\nÁudios por classe:")
print(audio_results["class_name"].value_counts().to_string())

audio_acc      = accuracy_score(audio_results["true_label"],
                                audio_results["pred_majority"])
audio_f1_macro = f1_score(audio_results["true_label"],
                           audio_results["pred_majority"],
                           average="macro", zero_division=0)

print("\n" + "=" * 55)
print("COMPARAÇÃO: segmento vs áudio — Wang 5-Stack CNN")
print("=" * 55)
print(f"Accuracy por SEGMENTO:                  {test_accuracy:.4f}")
print(f"Accuracy por ÁUDIO (voto majoritário):  {audio_acc:.4f}")
print(f"F1 macro  por ÁUDIO:                    {audio_f1_macro:.4f}")
print("=" * 55)

# Per-audio confusion matrix
audio_cm = confusion_matrix(audio_results["true_label"],
                              audio_results["pred_majority"])

plt.figure(figsize=(7, 6))
sns.heatmap(audio_cm, annot=True, fmt="d", cmap="Greens", cbar=True,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size": 14})
plt.xlabel("Predito (voto majoritário dos 11 segmentos)", fontsize=11)
plt.ylabel("Real", fontsize=12)
plt.title("Matriz de Confusão POR ÁUDIO — Wang 5-Stack CNN — Test Set", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nConcordância média entre os 11 segmentos do mesmo áudio: "
      f"{audio_results['agreement'].mean():.4f}")
print("(Próximo de 1.0 = predições consistentes; próximo de 0.33 = caos.)")

print("\nClassification report POR ÁUDIO:")
print(classification_report(audio_results["true_label"],
                            audio_results["pred_majority"],
                            target_names=CLASS_NAMES, digits=4))


In [ ]:
# Save misclassified segments for manual inspection
errors_df = test_df[test_df["label"] != test_df["pred"]].copy()
cols = ["path", "label", "pred", "class_name", "audio_id"] \
       + [f"prob_{n}" for n in CLASS_NAMES]
errors_df = errors_df[cols]
errors_df.to_csv("test_errors_wang_3class.csv", index=False)

print(f"Total de erros no test (segmentos): {len(errors_df)} / {len(test_df)}")
print(f"Taxa de erro: {len(errors_df)/len(test_df)*100:.2f}%")
print(f"Salvos em: test_errors_wang_3class.csv\n")

print("Distribuição dos erros por classe REAL:")
print(errors_df["class_name"].value_counts().to_string())

print("\nDistribuição dos erros por classe PREDITA:")
err_pred_names = errors_df["pred"].map(dict(enumerate(CLASS_NAMES)))
print(err_pred_names.value_counts().to_string())


In [ ]:
np.savez("data_wang_3class.npz",
         y_true=y_true.astype(np.int8),
         y_probs=y_probs.astype(np.float32),
         audio_id=test_df["audio_id"].values.astype(str))
print("ok | segmentos:", y_probs.shape)